In [1]:
import os
import json
import warnings
import numpy as np
import torch
from tqdm import tqdm
from collections import defaultdict
from transformers import AutoTokenizer, AutoModel

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

DATA_PATH  = "../../dataset/reveal/dataset.json"
OUTPUT_DIR = "./codet5"
MODEL_NAME = "Salesforce/codet5p-110m-embedding"
MAX_LEN    = 512
BATCH_SIZE = 8

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

def load_dataset(path):
    print(f"Loading dataset from {path} ...")
    with open(path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    grouped = defaultdict(list)
    for item in raw_data:
        grouped[item["project"]].append(item)
    print(f"Projects found: {list(grouped.keys())}")
    return grouped

def get_codet5_model():
    print(f"Loading CodeT5+ model ({MODEL_NAME}) on {DEVICE} ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model     = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE)
    model.eval()
    return tokenizer, model

def embed_code_snippets(snippets, tokenizer, model):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(snippets), BATCH_SIZE)):
            batch     = snippets[i:i + BATCH_SIZE]
            input_ids = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LEN
            ).input_ids.to(DEVICE)
            embeddings = model(input_ids)
            all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

def save_embeddings(project, embeddings, labels):
    tag       = project.lower()
    emb_path  = os.path.join(OUTPUT_DIR, f"{tag}_codet5p_embeddings.npy")
    lab_path  = os.path.join(OUTPUT_DIR, f"{tag}_codet5p_labels.npy")
    meta_path = os.path.join(OUTPUT_DIR, f"{tag}_codet5p_meta.json")
    np.save(emb_path, embeddings)
    np.save(lab_path, labels)
    meta = {"model": MODEL_NAME, "max_len": MAX_LEN, "pooling": "none", "device": DEVICE}
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"Saved {project}: {embeddings.shape} -> {emb_path}")

if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    combined_data    = load_dataset(DATA_PATH)
    tokenizer, model = get_codet5_model()

    for project_name, samples in combined_data.items():
        print(f"\nProcessing project: {project_name}")
        codes  = [s["code"] for s in samples]
        labels = np.array([s["label"] for s in samples])

        lengths   = [len(tokenizer.encode(c)) for c in codes]
        truncated = sum(1 for l in lengths if l > MAX_LEN)
        if truncated:
            print(f"{truncated}/{len(codes)} samples exceed {MAX_LEN} tokens and will be truncated")

        embeddings = embed_code_snippets(codes, tokenizer, model)
        save_embeddings(project_name, embeddings, labels)

    print("\nDone - all CodeT5+ embeddings generated successfully.")

/opt/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset from ../../dataset/reveal/dataset.json ...
Projects found: ['chrome', 'debian']
Loading CodeT5+ model (Salesforce/codet5p-110m-embedding) on mps ...


A new version of the following files was downloaded from https://huggingface.co/Salesforce/codet5p-110m-embedding:
- configuration_codet5p_embedding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/Salesforce/codet5p-110m-embedding:
- modeling_codet5p_embedding.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Token indices sequence length is longer than the specified maximum sequence length for this model (1572 > 512). Running this sequence through the model will result in indexing errors



Processing project: chrome
1100/4436 samples exceed 512 tokens and will be truncated


100%|█████████████████████████████████████████████████| 555/555 [05:49<00:00,  1.59it/s]


Saved chrome: (4436, 256) -> ./codet5/chrome_codet5p_embeddings.npy

Processing project: debian
4169/18298 samples exceed 512 tokens and will be truncated


100%|███████████████████████████████████████████████| 2288/2288 [25:36<00:00,  1.49it/s]

Saved debian: (18298, 256) -> ./codet5/debian_codet5p_embeddings.npy

Done - all CodeT5+ embeddings generated successfully.
